# CatAPI RAG Inspection

Учебный notebook для ручной инспекции CatAPI RAG-пайплайна проекта `cat-breed-assistant`.

Путь, который проверяем:

```text
CatAPI → raw data → processed breed documents → chunks → embeddings → ChromaDB → retrieval → LLM answer
```

Важно: это текстовый assistant. Пользователь пишет текстовый вопрос, сервис отвечает текстом. Картинки породы можно показывать, если CatAPI даёт `image_url` или `reference_image_id`, но CV/image classification здесь не делаем.

## Environment setup

Перед запуском notebook установите зависимости из корня проекта:

```bash
pip install -r requirements.txt
```

Если notebook запускается в Kaggle и зависимости были переустановлены, сохраните notebook и сделайте restart kernel/session. Это особенно важно для `numpy`, `torch`, `sentence-transformers` и `chromadb`.

Notebook не переустанавливает тяжёлые зависимости автоматически. Если модель embeddings не найдена локально, ячейка sanity check покажет, что нужно либо построить индекс скриптом, либо временно поставить `ALLOW_MODEL_DOWNLOAD = True`.

## 1. Цель notebook

Этот notebook нужен, чтобы руками увидеть и проверить каждый слой CatAPI RAG:

- что пришло из CatAPI;
- как raw data превращается в документы;
- как документы превращаются в chunks;
- как chunks кодируются embeddings;
- что лежит в ChromaDB;
- какие chunks достаются по английским и русским запросам;
- как retrieved context попадает в финальный prompt и LLM/mock answer.

## 2. Setup

Импортируем стандартные библиотеки, находим root проекта и проверяем ожидаемые пути.

В Kaggle удобно клонировать репозиторий в `/kaggle/working/cat-breed-assistant`. Если проект лежит в другом месте, задайте переменную окружения `CAT_BREED_ASSISTANT_ROOT`.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from pprint import pprint

try:
    import pandas as pd
except Exception as exc:
    pd = None
    print(f"pandas недоступен, буду использовать fallback display: {exc}")

try:
    from IPython.display import display
except Exception:
    display = None


def show_rows(rows, limit=10):
    rows = list(rows or [])[:limit]
    if pd is not None and display is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            pprint(row)


def preview_text(text, n=300):
    text = " ".join(str(text or "").split())
    return text if len(text) <= n else text[:n].rstrip() + "..."


def find_project_root() -> Path:
    env_root = os.getenv("CAT_BREED_ASSISTANT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path.cwd(),
        Path.cwd().parent,
        Path('/kaggle/working/cat-breed-assistant'),
        Path('/kaggle/working'),
    ])

    for base in candidates:
        candidate = base.resolve()
        if (candidate / 'app.py').exists() and (candidate / 'data').exists():
            return candidate

    raise RuntimeError(
        'Не найден root проекта cat-breed-assistant. '
        'В Kaggle клонируйте репозиторий в /kaggle/working/cat-breed-assistant '
        'или задайте CAT_BREED_ASSISTANT_ROOT=/path/to/cat-breed-assistant.'
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

PATHS = {
    'raw_catapi': PROJECT_ROOT / 'data/raw/catapi_breeds.json',
    'documents': PROJECT_ROOT / 'data/processed/catapi_breed_documents.jsonl',
    'chunks': PROJECT_ROOT / 'data/processed/catapi_chunks.jsonl',
    'chroma': PROJECT_ROOT / 'data/chroma',
}

for name, path in PATHS.items():
    print(f'{name:12} exists={path.exists()} path={path}')

## 3. Inspect raw CatAPI breeds

Загружаем `data/raw/catapi_breeds.json`, если он уже есть.

Если файла нет, сначала выполните:

```bash
python scripts/fetch_catapi_breeds.py
```

In [ ]:
raw_breeds = []
raw_path = PATHS['raw_catapi']

if not raw_path.exists():
    print('Raw CatAPI file not found.')
    print('Run: python scripts/fetch_catapi_breeds.py')
else:
    raw_breeds = json.loads(raw_path.read_text(encoding='utf-8'))
    print('Количество пород:', len(raw_breeds))
    print('Первые 5 пород:', [breed.get('name') for breed in raw_breeds[:5]])
    print('Ключи первой записи:')
    pprint(sorted(raw_breeds[0].keys()) if raw_breeds else [])

In [ ]:
def find_raw_breed(name):
    target = name.lower()
    for breed in raw_breeds:
        if str(breed.get('name', '')).lower() == target:
            return breed
    return None

interesting_breeds = ['British Shorthair', 'Maine Coon', 'Siamese', 'Sphynx']
raw_examples = []
for name in interesting_breeds:
    breed = find_raw_breed(name)
    if not breed:
        raw_examples.append({'name': name, 'found': False})
        continue
    raw_examples.append({
        'id': breed.get('id'),
        'name': breed.get('name'),
        'origin': breed.get('origin'),
        'temperament': breed.get('temperament'),
        'description': preview_text(breed.get('description'), 220),
        'life_span': breed.get('life_span'),
        'wikipedia_url': breed.get('wikipedia_url'),
        'reference_image_id': breed.get('reference_image_id'),
    })

show_rows(raw_examples, limit=10)

## 4. Inspect processed documents

Читаем `data/processed/catapi_breed_documents.jsonl`.

Если файла нет, выполните:

```bash
python scripts/build_catapi_documents.py
```

In [ ]:
def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


documents = read_jsonl(PATHS['documents'])
if not documents:
    print('Processed documents not found.')
    print('Run: python scripts/build_catapi_documents.py')
else:
    print('Количество документов:', len(documents))
    print('Поля документа:', sorted(documents[0].keys()))
    print('Metadata keys:', sorted(documents[0].get('metadata', {}).keys()))

    doc_preview = []
    for doc in documents[:5]:
        doc_preview.append({
            'doc_id': doc.get('doc_id'),
            'breed_id': doc.get('breed_id'),
            'breed_name': doc.get('breed_name'),
            'source': doc.get('source'),
            'metadata': doc.get('metadata'),
            'text_preview': preview_text(doc.get('text'), 260),
        })
    show_rows(doc_preview, limit=5)

## 5. Inspect chunks

Читаем `data/processed/catapi_chunks.jsonl`.

В первой версии chunking максимально простой: один chunk = один профиль породы.

In [ ]:
chunks = read_jsonl(PATHS['chunks'])
if not chunks:
    print('Chunks not found.')
    print('Run: python scripts/build_catapi_chunks.py')
else:
    print('Количество chunks:', len(chunks))
    print('Поля chunk:', sorted(chunks[0].keys()))
    print('Metadata keys:', sorted(chunks[0].get('metadata', {}).keys()))

    chunk_preview = []
    for chunk in chunks[:5]:
        chunk_preview.append({
            'chunk_id': chunk.get('chunk_id'),
            'breed_id': chunk.get('breed_id'),
            'breed_name': chunk.get('breed_name'),
            'origin': chunk.get('metadata', {}).get('origin'),
            'text_preview': preview_text(chunk.get('text'), 240),
        })
    show_rows(chunk_preview, limit=5)

    print('Породы, доступные для retrieval:')
    print(', '.join(sorted(chunk.get('breed_name', '') for chunk in chunks if chunk.get('breed_name'))))

## 6. Embedding sanity check

Пробуем загрузить `sentence-transformers/all-MiniLM-L6-v2` и проверить, что embeddings считаются.

По умолчанию:

```python
ALLOW_MODEL_DOWNLOAD = False
```

Это значит: сначала ищем модель локально. Если её нет в cache, ячейка не падает, а показывает понятное сообщение.

In [ ]:
ALLOW_MODEL_DOWNLOAD = False
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = None

try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(
        MODEL_NAME,
        local_files_only=not ALLOW_MODEL_DOWNLOAD,
    )
    print('Embedding model loaded:', MODEL_NAME)
except Exception as exc:
    print('Embedding model is not available locally.')
    print('Reason:', exc)
    print('Options:')
    print('1. Run: python scripts/build_catapi_rag_index.py')
    print('2. Or set ALLOW_MODEL_DOWNLOAD = True and rerun this cell if internet/model download is allowed.')


def cosine_similarity(vec_a, vec_b):
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if not norm_a or not norm_b:
        return 0.0
    return dot / (norm_a * norm_b)

if embedding_model is not None and chunks:
    sample_chunks = chunks[:3]
    sample_texts = [chunk['text'] for chunk in sample_chunks]
    sample_embeddings = embedding_model.encode(sample_texts)
    print('Embedding shape:', sample_embeddings.shape)

    queries = [
        'round face dense plush coat',
        'large gentle cat',
        'hairless cat care',
    ]
    for query in queries:
        query_embedding = embedding_model.encode([query])[0]
        scores = []
        for chunk, embedding in zip(sample_chunks, sample_embeddings):
            scores.append({
                'query': query,
                'breed_name': chunk.get('breed_name'),
                'cosine_similarity': round(float(cosine_similarity(query_embedding, embedding)), 4),
                'text_preview': preview_text(chunk.get('text'), 140),
            })
        print('\nQuery:', query)
        show_rows(sorted(scores, key=lambda row: row['cosine_similarity'], reverse=True), limit=3)
else:
    print('Embedding sanity check skipped: model or chunks are not available.')

## 7. Build / inspect ChromaDB index

Индекс строится отдельным скриптом:

```bash
python scripts/build_catapi_rag_index.py
```

Скрипт читает `data/processed/catapi_chunks.jsonl`, считает embeddings через `sentence-transformers/all-MiniLM-L6-v2` и сохраняет ChromaDB index в `data/chroma/`, collection name: `cat_breed_chunks`.

In [ ]:
CHROMA_COLLECTION = 'cat_breed_chunks'
chroma_client = None
chroma_collection = None

try:
    import chromadb
    if PATHS['chroma'].exists():
        chroma_client = chromadb.PersistentClient(path=str(PATHS['chroma']))
        collections = chroma_client.list_collections()
        collection_names = [getattr(collection, 'name', collection) for collection in collections]
        print('Collections:', collection_names)
        if CHROMA_COLLECTION in collection_names:
            chroma_collection = chroma_client.get_collection(CHROMA_COLLECTION)
            print('Collection count:', chroma_collection.count())
            try:
                peek = chroma_collection.peek(3)
                print('Sample metadata:')
                pprint(peek.get('metadatas', []))
            except Exception as exc:
                print('Could not peek collection:', exc)
        else:
            print('CatAPI Chroma collection not found.')
            print('Run: python scripts/build_catapi_rag_index.py')
    else:
        print('Chroma path not found.')
        print('Run: python scripts/build_catapi_rag_index.py')
except Exception as exc:
    print('Could not import or inspect ChromaDB:', exc)

## 8. Manual retrieval

Делаем retrieval напрямую из ChromaDB.

Важно: collection был создан с готовыми embeddings, поэтому для query мы тоже считаем embedding локально и вызываем `query_embeddings`.

In [ ]:
def retrieve_from_chroma(query: str, top_k: int = 5):
    if chroma_collection is None:
        print('Chroma collection is not available. Run: python scripts/build_catapi_rag_index.py')
        return []
    if embedding_model is None:
        print('Embedding model is not available, retrieval skipped.')
        return []

    query_embedding = embedding_model.encode([query])[0].tolist()
    result = chroma_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances'],
    )

    rows = []
    ids = result.get('ids', [[]])[0]
    documents_result = result.get('documents', [[]])[0]
    metadatas = result.get('metadatas', [[]])[0]
    distances = result.get('distances', [[]])[0]

    for rank, (chunk_id, text, metadata, distance) in enumerate(
        zip(ids, documents_result, metadatas, distances), start=1
    ):
        metadata = metadata or {}
        rows.append({
            'rank': rank,
            'chunk_id': chunk_id,
            'distance': round(float(distance), 4),
            'breed_name': metadata.get('breed_name'),
            'breed_id': metadata.get('breed_id'),
            'text_preview': preview_text(text, 220),
            'wikipedia_url': metadata.get('wikipedia_url'),
            'image_url': metadata.get('image_url') or metadata.get('reference_image_id'),
            'metadata': metadata,
            'text': text,
        })
    return rows

english_queries = [
    'British Shorthair round face dense coat',
    'Maine Coon large gentle cat',
    'Sphynx hairless cat care',
    'Siamese vocal intelligent cat',
]

english_results = {}
for query in english_queries:
    print('\nQUERY:', query)
    rows = retrieve_from_chroma(query, top_k=5)
    english_results[query] = rows
    show_rows([{k: v for k, v in row.items() if k not in {'metadata', 'text'}} for row in rows], limit=5)

## 9. Russian query debug

Проверяем русские запросы и сравниваем их с английскими. Модель `all-MiniLM-L6-v2` обычно лучше работает по английскому, поэтому русские запросы могут доставать менее точные chunks.

In [ ]:
russian_queries = [
    'британская кошка круглая морда плотная шерсть',
    'мейн кун большой спокойный кот',
    'как ухаживать за сфинксом',
    'сиамская кошка разговорчивая',
]

russian_results = {}
for query in russian_queries:
    print('\nQUERY:', query)
    rows = retrieve_from_chroma(query, top_k=5)
    russian_results[query] = rows
    show_rows([{k: v for k, v in row.items() if k not in {'metadata', 'text'}} for row in rows], limit=5)

### Вывод по русским запросам

Если русские запросы возвращают менее точные породы, это ожидаемо для англоязычной embedding-модели. Позже можно улучшить retrieval одним из способов:

- добавить multilingual embedding model;
- переводить query на английский перед retrieval;
- хранить русские aliases/описания рядом с CatAPI chunks;
- использовать гибридный поиск: aliases + vector retrieval.

## 10. Final RAG prompt inspection

Берём вопрос:

```text
Чем британские котики отличаются от обычных?
```

Достаём chunks и собираем prompt, который можно отправить в LLM.

In [ ]:
question = 'Чем британские котики отличаются от обычных?'
retrieval_query = 'British Shorthair round face dense coat temperament care'
retrieved_rows = retrieve_from_chroma(retrieval_query, top_k=3)

system_instruction = '\n'.join([
    'Ты дружелюбный ассистент про породы кошек.',
    'Отвечай на русском языке, понятно и без слишком научного тона.',
    'Используй только факты из CatAPI context.',
    'Не выдумывай медицинские рекомендации.',
    'Если данных мало, честно скажи, что в контексте информации мало.',
])

context_blocks = []
for row in retrieved_rows:
    context_blocks.append(
        f"Breed: {row.get('breed_name')}\n"
        f"Source: thecatapi\n"
        f"Wikipedia: {row.get('wikipedia_url')}\n"
        f"Text:\n{row.get('text')}"
    )

joined_context = '\n\n---\n\n'.join(context_blocks)

prompt = (
    f"SYSTEM:\n{system_instruction}\n\n"
    f"RETRIEVED CATAPI CONTEXT:\n{joined_context}\n\n"
    f"USER QUESTION:\n{question}"
)

print(prompt)

## 11. LLM answer test

Если есть `MISTRAL_API_KEY`, пробуем вызвать Mistral provider из проекта и передать `retrieved_context`.

Если ключа нет, notebook не падает: показываем mock answer на основе retrieved chunks.

In [ ]:
from dotenv import load_dotenv
from src.breed_retriever import build_breed_context
from src.cat_knowledge import generate_mock_answer

load_dotenv()

retrieved_context = []
for row in retrieved_rows:
    metadata = row.get('metadata') or {}
    retrieved_context.append({
        'text': row.get('text'),
        'distance': row.get('distance'),
        'metadata': {
            'breed': metadata.get('breed_name'),
            'breed_name': metadata.get('breed_name'),
            'breed_id': metadata.get('breed_id'),
            'source_id': row.get('chunk_id'),
            'source': metadata.get('source'),
            'wikipedia_url': metadata.get('wikipedia_url'),
            'image_url': metadata.get('image_url'),
            'reference_image_id': metadata.get('reference_image_id'),
        },
    })

breed_context = build_breed_context(question)

if os.getenv('MISTRAL_API_KEY'):
    try:
        from src.mistral_client import generate_mistral_answer
        answer = generate_mistral_answer(question, breed_context, retrieved_context)
        print(answer)
    except Exception as exc:
        print('Mistral call failed, showing mock answer instead.')
        print('Reason:', exc)
        print(generate_mock_answer(question, breed_context, retrieved_context))
else:
    print('MISTRAL_API_KEY is not set. Showing mock answer based on retrieved chunks.')
    print(generate_mock_answer(question, breed_context, retrieved_context))

## Финальная заметка

Этот notebook проверяет CatAPI RAG руками и не меняет backend. Следующий engineering-шаг — подключить готовый Chroma retriever к FastAPI service layer, сохранив fallback, если индекс ещё не построен.